In [ ]:
import torch
import numpy as np
from torch.optim import LBFGS
import matplotlib.pyplot as plt

class GameMinimizer:
    def __init__(self, num_players, num_actions, payoff_tensor):
        """
        :param num_players: int
        :param num_actions: list of int (nombre d'actions par joueur)
        :param payoff_tensor: torch.Tensor de dimension (num_players, action_p1, action_p2, ...)
        """
        self.num_players = num_players
        self.num_actions = num_actions
        self.payoff_tensor = payoff_tensor
        
        # Initialisation des stratégies (paramètres optimisables)
        # On initialise avec une distribution uniforme + un peu de bruit
        self.strategies = []
        for n in num_actions:
            s = torch.rand(n) # n uniformes
            s = s / s.sum() # chaque ligne deviens une distribution
            s.requires_grad_(True) # Permettra de calculer le gradient
            self.strategies.append(s) #tenseur avec toutes les stratégies
        
        self.history = []

    def compute_utility(self, strategies_list, player_idx):
        """
        Calcule l'espérance du gain u(i) pour le joueur i.
        u(i) = sum_{a} [ P(a) * Payoff_i(a) ]
        """
        # On crée le produit extérieur des probabilités pour obtenir la probabilité de chaque profil d'action
        # On calcul la probabilité de chaque cas (produit de toutes les probabilités de l'action de chaque joueur car indépendance) puis multiplication avec le payoff/gain actions pures
        prob_matrix = strategies_list[0]
        for i in range(1, self.num_players):
            prob_matrix = torch.matmul(prob_matrix.unsqueeze(-1), strategies_list[i].unsqueeze(0)) #multiplication(matmul) entre une vecteur colonne (unsqueeze(-1)) et ligne (unsqueeze(0)) => effectue le produit exterieur => matrice des probabilités d'intersection
            prob_matrix = prob_matrix.view(-1) # On aplatit pour continuer le produit extérieur si > 2 joueurs (grosse matrice qu'on applatie après)
        
        # Reshape pour correspondre au tenseur de gains
        prob_matrix = prob_matrix.view(self.payoff_tensor.shape[1:]) # 1: car payooff_tensor a sa premiàre dimension qui correspond au joueur pour qui les gains vont etre, donc on enlève la dimension nombre de joueurs payoff_tensor ne change pas de taille selon le joueur
        #1ere dimension joueur 0 2eme dimension joueur 1 .... 
        
        # Gain espéré : somme pondérée des gains par les probabilités
        return torch.sum(prob_matrix * self.payoff_tensor[player_idx])

    def compute_loss(self):
        """ Calcule la fonction F : somme des regrets + pénalité de distribution """
        probs = self.strategies
        total_regret = 0
        penalty = 0
        
        for i in range(self.num_players):
            # 1. Calcul de l'utilité actuelle u(i)(x_i, x_others) v   => le gain du joueur i selon ses décisions actuelles
            u_current = self.compute_utility(probs, i)
            
            # 2. Calcul des regrets par rapport aux actions pures k
            for k in range(self.num_actions[i]):
                # Stratégie où le joueur i joue l'action pure k  P(joueur joue k) = 1, 0 sinon
                pure_strategy_i = torch.zeros(self.num_actions[i])
                pure_strategy_i[k] = 1.0
                
                probs_with_pure = [p for p in probs] 
                probs_with_pure[i] = pure_strategy_i # On modifie juste la stratégie du joueur i
                
                u_pure = self.compute_utility(probs_with_pure, i) # utilité avec stratégie pure
                
                # g_i = u_current - u_pure
                # On veut minimiser le regret de ne pas avoir joué k : max(0, u_pure - u_current)^2
                regret_k = torch.clamp(u_pure - u_current, min=0)**2
                total_regret += regret_k
            
            # 3. Pénalité pour non-respect de la loi de probabilité
            #La somme fait 1
            sum_prob = torch.sum(probs[i])
            penalty += 1e6 * (sum_prob - 1)**2
            # et les valeurs sont positives
            penalty += 1e6 * torch.sum(torch.clamp(-probs[i], min=0)**2)

        return total_regret + penalty

    def solve(self, iterations=100):
        optimizer = LBFGS(self.strategies, lr=1, max_iter=20, history_size=10)

        def closure():
            optimizer.zero_grad()
            loss = self.compute_loss()
            loss.backward()
            return loss

        for i in range(iterations):
            current_probs = [p.detach().clone() for p in self.strategies] # detach permet de redevenir un simple tableau sans tracker des opérations pour le gradient 
            self.history.append(current_probs)
            loss = optimizer.step(closure)
            if i % 10 == 0:
                print(f"Itération {i}, Loss: {loss.item():.6f}")

        return [s.detach().numpy() for s in self.strategies] # detacher du gradient les distributions
    
    def plot_evolution(self):
    
        # Conversion de l'historique en array numpy pour faciliter le slicing
        # Format : (Nb_iterations, Nb_joueurs, Nb_actions)
        hist = np.array([[p.numpy() for p in step] for step in self.history])
        iterations = range(len(hist))
        
        plt.figure(figsize=(12, 6))
        
        for i in range(self.num_players):
            for j in range(self.num_actions[i]):
                plt.plot(iterations, hist[:, i, j], 
                        label=f"Joueur {i+1} - Action {j+1}")
        
        plt.title("Évolution des probabilités de stratégie (Minimisation du Regret)")
        plt.xlabel("Itérations (Pas BFGS)")
        plt.ylabel("Probabilité")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    


## Jeu Pierre Feuille Papier Ciseaux

In [ ]:
# Configuration du jeu : Pierre-Papier-Ciseaux
# Gains pour Joueur 0 (Matrice de gain)
# P1 \ P2 | Pierre | Papier | Ciseaux
# Pierre   |    0   |   -1   |    1
# Papier   |    1   |    0   |   -1
# Ciseaux  |   -1   |    1   |    0

payoffs_p0 = torch.tensor([
    [0., -1., 1.],
    [1., 0., -1.],
    [-1., 1., 0.]
])
payoffs_p1 = -payoffs_p0  # Somme nulle

# Tenseur global (2 joueurs, actions [3, 3])
game_tensor = torch.stack([payoffs_p0, payoffs_p1])

# Initialisation et résolution
game = GameMinimizer(num_players=2, num_actions=[3, 3], payoff_tensor=game_tensor)
final_strategies = game.solve(iterations=100)
game.plot_evolution()
print("\nDistributions finales :")
for i, strat in enumerate(final_strategies):
    # On normalise juste pour l'affichage final
    strat_norm = np.abs(strat) / np.sum(np.abs(strat))
    print(f"Joueur {i}: {strat_norm}")



## Gardien et tireur

In [ ]:
# Matrice de gain pour le Gardien (P0)
# Colonnes = Gardien (G, D) | Lignes = Tireur (G, D)
payoffs_p0 = torch.tensor([
    [ 2., -1.],  # Tireur tire à Gauche
    [-1.,  3.]   # Tireur tire à Droite
], dtype=torch.float64)

payoffs_p1 = -payoffs_p0  # Somme nulle

# Tenseur global [2 joueurs, 2 actions, 2 actions]
game_tensor_penalty = torch.stack([payoffs_p0, payoffs_p1])

# Création de l'instance pour le Penalty
penalty_game = GameMinimizer(
    num_players=2, 
    num_actions=[2, 2], 
    payoff_tensor=game_tensor_penalty
)

# Lancement de la résolution
# On surveille le gradient pour voir si L-BFGS "travaille" bien
final_strats = penalty_game.solve(iterations=10)

penalty_game.plot_evolution()

for i, strat in enumerate(final_strats):
    # On normalise juste pour l'affichage final
    strat_norm = np.abs(strat) / np.sum(np.abs(strat))
    print(f"Joueur {i}: {strat_norm}")

## Jeu de coopération

In [ ]:
# Matrice de gain pour le Gardien (P0)
# Colonnes = Gardien (G, D) | Lignes = Tireur (G, D)
payoffs_p0 = torch.tensor([
    [ 2., -1.],  # Tireur tire à Gauche
    [-1.,  3.]   # Tireur tire à Droite
], dtype=torch.float64)

payoffs_p1 = payoffs_p0 

# Tenseur global [2 joueurs, 2 actions, 2 actions]
game_tenseur_cooperation = torch.stack([payoffs_p0, payoffs_p1])

# Création de l'instance pour le Penalty
game_cooperation = GameMinimizer(
    num_players=2, 
    num_actions=[2, 2], 
    payoff_tensor=game_tenseur_cooperation
)

# Lancement de la résolution
# On surveille le gradient pour voir si L-BFGS "travaille" bien
final_strats = game_cooperation.solve(iterations=10)

penalty_game.plot_evolution()

for i, strat in enumerate(final_strats):
    # On normalise juste pour l'affichage final
    strat_norm = np.abs(strat) / np.sum(np.abs(strat))
    print(f"Joueur {i}: {strat_norm}")